# Deutsch–Jozsa classification

Distinguish a balanced oracle from a constant oracle using exact output probabilities.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the quantum problem

Deutsch-Jozsa distinguishes constant and balanced Boolean oracles with a single quantum query in the ideal model.

In [2]:
def deutsch_jozsa(balanced):
    n = 4
    circuit = QuantumCircuit(n + 1)
    circuit.x(n)
    circuit.h(range(n + 1))
    if balanced:
        for wire in range(n):
            circuit.cx(wire, n)
    circuit.h(range(n))
    return circuit

circuits = [deutsch_jozsa(False), deutsch_jozsa(True)]

def reference_probabilities():
    return np.asarray([Statevector.from_instruction(c).probabilities(qargs=range(4)) for c in circuits])

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(reference_probabilities)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
backend = MettleQBackend(method="statevector", device="cpu")
compiled = [transpile(c, backend, optimization_level=1) for c in circuits]

def mettleq_probabilities():
    values = []
    for circuit in compiled:
        state = backend.run(circuit, shots=1, return_statevector=True).result().data(0)["statevector"]
        values.append(Statevector(state).probabilities(qargs=range(4)))
    return np.asarray(values)

candidate, mettleq_ms, _ = benchmark(mettleq_probabilities)
error = max_abs_error(reference, candidate)
classes = [int(np.argmax(row) != 0) for row in candidate]
method, device = qiskit_selection(backend)

## 4. Check correctness before discussing speed

The probability vectors must agree numerically and the inferred oracle class must match exactly.

In [5]:
tutorial_result = emit_result(
    notebook="qiskit/05_deutsch_jozsa.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="probability vector atol=2e-6 and exact oracle class",
    passed=error <= 2e-6 and classes == [0, 1],
    exact_match=classes == [0, 1],
    selected_method=method,
    selected_device=device,
    metrics={"max_probability_error": error, "classifications": classes},
)


Comparison summary
------------------
Correctness contract: PASS — probability vector atol=2e-6 and exact oracle class
SDK reference median: 0.626 ms
MettleQ median:       1.297 ms
Timing interpretation: the SDK reference was 2.074x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: yes

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "probability vector atol=2e-6 and exact oracle class", "exact_match": true, "framework": "qiskit", "machine": "arm64", "metrics": {"classifications": [0, 1], "max_probability_error": 2.0281592438831098e-07}, "mettleq_median_ms": 1.2974999845027924, "notebook": "qiskit/05_deutsch_jozsa.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 0.6257499917410314, "reference_over_mettleq": 0.4822736024777846, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}


## What should you conclude?

This is an algorithm tutorial and regression test. Four qubits are far below any GPU crossover.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.